<a href="https://colab.research.google.com/github/thuynhungp068-design/cuda_learning_with_google_colab/blob/main/cuda_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile hello.cu

#include <cstdio>

__global__ void hello_kernel(){
  printf("Hello from GPU! block=%d,thread=%d\n",blockIdx.x,threadIdx.x);
}

int main(){
  hello_kernel<<<2,4>>>();
  cudaDeviceSynchronize();
  return 0;
}

Writing hello.cu


In [ ]:
!nvcc -o hello hello.cu

In [ ]:
!./hello

Hello from GPU! block=1,thread=0
Hello from GPU! block=1,thread=1
Hello from GPU! block=1,thread=2
Hello from GPU! block=1,thread=3
Hello from GPU! block=0,thread=0
Hello from GPU! block=0,thread=1
Hello from GPU! block=0,thread=2
Hello from GPU! block=0,thread=3


In [ ]:
%%writefile vector_add.cu

#include <cstdio>

__global__ void vector_add(const float* a,const float* b,float* result,int n){
  int i=blockIdx.x*blockDim.x+threadIdx.x;
  if(i<n){
    result[i]=a[i]+b[i];
  }
}

int main(){
  const int N=1024;
  const size_t size=N*sizeof(float);

  float h_a[N],h_b[N],h_result[N];
  for(int i=0;i<N;i++){
    h_a[i]=(float)i;
    h_b[i]=(float)N-i;
  }

  float *d_a,*d_b,*d_result;
  cudaMalloc(&d_a,size);
  cudaMalloc(&d_b,size);
  cudaMalloc(&d_result,size);

  cudaMemcpy(d_a,h_a,size,cudaMemcpyHostToDevice);
  cudaMemcpy(d_b,h_b,size,cudaMemcpyHostToDevice);

  int threads_per_block=256;
  int blocks=(N+threads_per_block-1)/threads_per_block;
  vector_add<<<blocks,threads_per_block>>>(d_a,d_b,d_result,N);

  cudaDeviceSynchronize();

  cudaMemcpy(h_result,d_result,size,cudaMemcpyDeviceToHost);

  bool correct=true;
  for(int i=0;i<N;i++){
    if(h_result[i]!=N){
      correct=false;
      printf("Error at i=%d: %f\\n",i,h_result[i]);
      break;
    }
  }
  printf("%s\\n", correct ? "Vector add PASSED!" : "Vector add FAILED!");

  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_result);
  return 0;
}

Writing vector_add.cu


In [ ]:
!nvcc -arch=sm_75 -o vector_add vector_add.cu

In [ ]:
!./vector_add


Vector add PASSED!\n

In [ ]:
!nvidia-smi -q | grep -E "SM|Memory|L2|Multiprocessor"

==============NVSMI LOG==============
    FB Memory Usage
    BAR1 Memory Usage
    Conf Compute Protected Memory Usage
        Memory                            : 0 %
        Memory Current Temp               : N/A
        Memory Max Operating Temp         : N/A
    GPU Memory Power Readings 
        SM                                : 1020 MHz
        Memory                            : 5000 MHz
        Memory                            : 5001 MHz
        Memory                            : 5001 MHz
        Memory                            : N/A
        SM                                : 1590 MHz
        Memory                            : 5001 MHz


In [ ]:
%%writefile block_mapping.cu

#include <cuda_runtime.h>
#include <stdio.h>

__device__ uint get_smid() {
    uint smid;
    asm volatile("mov.u32 %0, %%smid;" : "=r"(smid));
    return smid;
}

__global__ void show_mapping() {
    if (threadIdx.x == 0) {
        printf("Block (%d,%d) -> SM %d\n",
            blockIdx.x, blockIdx.y, get_smid());
    }
}

int main() {
    show_mapping<<<80, 32>>>();
    cudaDeviceSynchronize();
    return 0;
}

Writing block_mapping.cu


In [ ]:
!nvcc -arch=sm_75 -o block_mapping block_mapping.cu

In [ ]:
%%writefile matrix_add.cu

// 檔案：matrix_add.cu
// 在 Colab 跑：
//   %%writefile matrix_add.cu
//   !nvcc -arch=sm_75 -O2 matrix_add.cu -o matrix_add && ./matrix_add

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) \
    do { cudaError_t e=(call); if(e!=cudaSuccess){ \
         fprintf(stderr,"CUDA error %s:%d %s\n",__FILE__,__LINE__,cudaGetErrorString(e)); \
         exit(1); } } while(0)

// C = A + B，A/B/C 都是 rows × cols 的矩陣（row-major）
__global__ void matrix_add(const float *A, const float *B, float *C,
                            int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // 邊界檢查：rows 和 cols 不一定是 blockDim 的整數倍
    if (row < rows && col < cols) {
        int idx = row * cols + col;
        C[idx] = A[idx] + B[idx];
    }
}

int main(void) {
    const int ROWS = 1023;  // 故意選非 16 倍數，測試邊界檢查
    const int COLS = 997;
    const size_t sz = (size_t)ROWS * COLS * sizeof(float);

    float *h_A = (float *)malloc(sz);
    float *h_B = (float *)malloc(sz);
    float *h_C = (float *)malloc(sz);
    for (int i = 0; i < ROWS * COLS; i++) {
        h_A[i] = (float)i;
        h_B[i] = (float)(ROWS * COLS - i);
    }

    float *d_A, *d_B, *d_C;
    CUDA_CHECK(cudaMalloc(&d_A, sz));
    CUDA_CHECK(cudaMalloc(&d_B, sz));
    CUDA_CHECK(cudaMalloc(&d_C, sz));
    CUDA_CHECK(cudaMemcpy(d_A, h_A, sz, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_B, h_B, sz, cudaMemcpyHostToDevice));

    // 2D block：16×16 = 256 個 thread（warp size 32 的整數倍，細節見下節）
    dim3 blockDim(16, 16);
    // Ceiling division：確保覆蓋所有 rows 和 cols
    dim3 gridDim((COLS + blockDim.x - 1) / blockDim.x,
                 (ROWS + blockDim.y - 1) / blockDim.y);

    printf("Matrix %dx%d, block %dx%d, grid %dx%d\n",
           ROWS, COLS, blockDim.y, blockDim.x, gridDim.y, gridDim.x);

    matrix_add<<<gridDim, blockDim>>>(d_A, d_B, d_C, ROWS, COLS);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(h_C, d_C, sz, cudaMemcpyDeviceToHost));

    // 驗證
    bool correct = true;
    float expected_sum = (float)(ROWS * COLS);  // A[i] + B[i] = i + (N-i) = N
    for (int i = 0; i < ROWS * COLS && correct; i++) {
        if (fabsf(h_C[i] - expected_sum) > 1e-3f) {
            printf("MISMATCH at i=%d: got %f, expected %f\n",
                   i, h_C[i], expected_sum);
            correct = false;
        }
    }
    printf("Result: %s\n", correct ? "CORRECT" : "WRONG");

    free(h_A); free(h_B); free(h_C);
    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_B));
    CUDA_CHECK(cudaFree(d_C));
    return 0;
}

Overwriting matrix_add.cu


In [ ]:
!nvcc -arch=sm_75 -o matrix_add matrix_add.cu

In [ ]:
!./matrix_add

Matrix 1023x997, block 16x16, grid 64x63
Result: CORRECT


In [ ]:

%%writefile grayscale.cu
//   !nvcc -arch=sm_75 -O2 grayscale.cu -o grayscale && ./grayscale

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) \
    do { cudaError_t e=(call); if(e!=cudaSuccess){ \
         fprintf(stderr,"CUDA error %s:%d %s\n",__FILE__,__LINE__,cudaGetErrorString(e)); \
         exit(1); } } while(0)

// Luma 係數（BT.601 標準，適合 sRGB 影像）
// Y = 0.299*R + 0.587*G + 0.114*B
// 這三個數字加起來 = 1.0，對人眼的感知亮度最準確
// 不選 (R+G+B)/3 是因為人眼對綠色最敏感、對藍色最不敏感
__global__ void rgb_to_gray(const unsigned char *rgb,  // 輸入：[width * height * 3]
                             unsigned char *gray,        // 輸出：[width * height]
                             int width, int height) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (col < width && row < height) {
        int pixel_idx = row * width + col;
        int rgb_idx   = pixel_idx * 3;  // R, G, B 連續儲存

        unsigned char r = rgb[rgb_idx + 0];
        unsigned char g = rgb[rgb_idx + 1];
        unsigned char b = rgb[rgb_idx + 2];

        // 用 float 計算，最後轉回 uint8
        float y = 0.299f * r + 0.587f * g + 0.114f * b;
        gray[pixel_idx] = (unsigned char)(y + 0.5f);  // 四捨五入
    }
}

int main(void) {
    const int W = 1920, H = 1080;  // Full HD
    const size_t rgb_sz  = (size_t)W * H * 3;
    const size_t gray_sz = (size_t)W * H;

    unsigned char *h_rgb  = (unsigned char *)malloc(rgb_sz);
    unsigned char *h_gray = (unsigned char *)malloc(gray_sz);

    // 用漸層填充模擬一張影像
    for (int row = 0; row < H; row++) {
        for (int col = 0; col < W; col++) {
            int idx = (row * W + col) * 3;
            h_rgb[idx+0] = (unsigned char)(col * 255 / W);     // R
            h_rgb[idx+1] = (unsigned char)(row * 255 / H);     // G
            h_rgb[idx+2] = 128;                                  // B constant
        }
    }

    unsigned char *d_rgb, *d_gray;
    CUDA_CHECK(cudaMalloc(&d_rgb,  rgb_sz));
    CUDA_CHECK(cudaMalloc(&d_gray, gray_sz));
    CUDA_CHECK(cudaMemcpy(d_rgb, h_rgb, rgb_sz, cudaMemcpyHostToDevice));

    // 2D launch：block 16×16，grid covering W×H
    dim3 blockDim(16, 16);   // 256 threads per block
    dim3 gridDim((W + 15) / 16, (H + 15) / 16);  // (120, 68)

    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0));
    CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));

    rgb_to_gray<<<gridDim, blockDim>>>(d_rgb, d_gray, W, H);

    CUDA_CHECK(cudaEventRecord(t1));
    CUDA_CHECK(cudaEventSynchronize(t1));
    float ms;
    CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));

    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(h_gray, d_gray, gray_sz, cudaMemcpyDeviceToHost));

    // 簡單驗證：左上角像素 (R=0, G=0, B=128) 的灰階值
    // Y = 0.299*0 + 0.587*0 + 0.114*128 = 14.592 ≈ 15
    printf("Pixel (0,0): R=%d G=%d B=%d → gray=%d (expected ~15)\n",
           h_rgb[0], h_rgb[1], h_rgb[2], h_gray[0]);
    printf("Kernel time: %.3f ms  (%.1f MP/s)\n",
           ms, (float)(W * H) / ms / 1000.0f);

    free(h_rgb); free(h_gray);
    CUDA_CHECK(cudaFree(d_rgb));
    CUDA_CHECK(cudaFree(d_gray));
    CUDA_CHECK(cudaEventDestroy(t0));
    CUDA_CHECK(cudaEventDestroy(t1));
    return 0;
}

Writing grayscale.cu


In [ ]:
!nvcc -arch=sm_75 -o grayscale grayscale.cu

In [ ]:
%%writefile transpose_naive.cu
//   !nvcc -arch=sm_75 -O2 transpose_naive.cu -o transpose_naive && ./transpose_naive

#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) \
    do { cudaError_t e=(call); if(e!=cudaSuccess){ \
         fprintf(stderr,"CUDA error %s:%d %s\n",__FILE__,__LINE__,cudaGetErrorString(e)); \
         exit(1); } } while(0)

// Naive 轉置：A 是 rows×cols 的矩陣，B 是 cols×rows 的轉置結果
// 讀 A 是 coalesced（同一 row 連續），寫 B 是 strided（不 coalesced）
// 這個版本在大矩陣時是 memory-bound 且有 coalescing 問題
// Ch 17 的 tiled 版本用 shared memory 解決這個問題
__global__ void transpose_naive(const float *A, float *B, int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;  // A 的 col
    int row = blockIdx.y * blockDim.y + threadIdx.y;  // A 的 row

    if (col < cols && row < rows) {
        // 讀 A[row][col]（row-major，連續 → coalesced）
        float val = A[row * cols + col];
        // 寫 B[col][row]（B 的 row 是 col，col 是 row → strided，不 coalesced）
        B[col * rows + row] = val;
    }
}

int main(void) {
    const int ROWS = 1024, COLS = 512;
    const size_t sz_A = (size_t)ROWS * COLS * sizeof(float);
    const size_t sz_B = (size_t)COLS * ROWS * sizeof(float);  // 轉置後 COLS×ROWS

    float *h_A = (float *)malloc(sz_A);
    float *h_B = (float *)malloc(sz_B);
    for (int i = 0; i < ROWS * COLS; i++) h_A[i] = (float)i;

    float *d_A, *d_B;
    CUDA_CHECK(cudaMalloc(&d_A, sz_A));
    CUDA_CHECK(cudaMalloc(&d_B, sz_B));
    CUDA_CHECK(cudaMemcpy(d_A, h_A, sz_A, cudaMemcpyHostToDevice));

    // Launch config：block 32×8（256 threads），x 方向對齊 warp
    dim3 blockDim(32, 8);
    dim3 gridDim((COLS + 31) / 32, (ROWS + 7) / 8);

    cudaEvent_t t0, t1;
    CUDA_CHECK(cudaEventCreate(&t0));
    CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventRecord(t0));
    transpose_naive<<<gridDim, blockDim>>>(d_A, d_B, ROWS, COLS);
    CUDA_CHECK(cudaEventRecord(t1));
    CUDA_CHECK(cudaEventSynchronize(t1));
    float ms;
    CUDA_CHECK(cudaEventElapsedTime(&ms, t0, t1));
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaMemcpy(h_B, d_B, sz_B, cudaMemcpyDeviceToHost));

    // 驗證：B[col][row] 應等於 A[row][col]
    bool correct = true;
    for (int r = 0; r < ROWS && correct; r++) {
        for (int c = 0; c < COLS && correct; c++) {
            float expected = h_A[r * COLS + c];
            float got      = h_B[c * ROWS + r];
            if (fabsf(got - expected) > 1e-6f) {
                printf("MISMATCH at (%d,%d): got %f expected %f\n", r, c, got, expected);
                correct = false;
            }
        }
    }
    printf("Result: %s\n", correct ? "CORRECT" : "WRONG");
    printf("Naive transpose: %.3f ms\n", ms);
    printf("(Ch 17 tiled version will be ~3-5x faster on this size)\n");

    free(h_A); free(h_B);
    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_B));
    CUDA_CHECK(cudaEventDestroy(t0));
    CUDA_CHECK(cudaEventDestroy(t1));
    return 0;
}

Writing transpose_naive.cu


In [ ]:
!nvcc -arch=sm_75 -o transpose_naive transpose_naive.cu